# Task 1 - Source Extraction

## KAUST Data Extraction

KAUST research data is collected from two sources:

- 2023: KAUST repository source file
- 2024–2025: Crossref REST API using the KAUST ROR identifier

Raw source data is stored in `data/raw/` without cleaning or transformation.

In [4]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime

### KAUST 2023 Repository Data

The 2023 KAUST dataset is provided as a raw repository CSV file and is retained without modification.

In [ ]:
raw_dir = Path("../data/raw")

kaust_2023_file = raw_dir / "KAUST_2023_raw.csv"

print("KAUST 2023 file exists:", kaust_2023_file.exists())
print("File:", kaust_2023_file)

KAUST 2023 file exists: True
File: ..\data\raw\KAUST_2023_raw.csv


: 

### KAUST 2024–2025 - Crossref API

Source: Crossref REST API

Endpoint: https://api.crossref.org/works

Institution: King Abdullah University of Science and Technology (KAUST)

KAUST ROR ID: 01q3tbs38

Publication period: 2024-01-01 to 2025-12-31

Authentication: No API key required.

The API response is saved as raw JSON without modification.

In [7]:
url = "https://api.crossref.org/works"

kaust_ror = "01q3tbs38"

date_filter = (
    f"ror-id:{kaust_ror},"
    "from-pub-date:2024-01-01,"
    "until-pub-date:2025-12-31"
)

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

print(date_filter)

ror-id:01q3tbs38,from-pub-date:2024-01-01,until-pub-date:2025-12-31


In [8]:
params = {
    "filter": date_filter,
    "rows": 5
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)

Status: 200
URL: https://api.crossref.org/works?filter=ror-id%3A01q3tbs38%2Cfrom-pub-date%3A2024-01-01%2Cuntil-pub-date%3A2025-12-31&rows=5


In [9]:
response.raise_for_status()

test_data = response.json()

total_results = test_data["message"]["total-results"]

print("Total KAUST records found:", total_results)

Total KAUST records found: 114


In [10]:
cursor = "*"
page = 1
total_records = 0

while True:
    params = {
        "filter": date_filter,
        "rows": 100,
        "cursor": cursor
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        page_data = response.json()
        items = page_data["message"]["items"]

        if not items:
            break

        output_file = raw_dir / f"KAUST_Crossref_2024_2025_page_{page}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(response.text)

        total_records += len(items)

        print(
            f"Page {page}: {len(items)} records | "
            f"Total: {total_records}"
        )

        cursor = page_data["message"].get("next-cursor")

        if not cursor:
            break

        if total_records >= page_data["message"]["total-results"]:
            break

        page += 1
        time.sleep(1.2)

    except requests.exceptions.Timeout:
        print("Request timed out.")
        break

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        break

Page 1: 100 records | Total: 100
Page 2: 14 records | Total: 114


In [11]:
print("Extraction completed.")
print("Total records downloaded:", total_records)

Extraction completed.
Total records downloaded: 114


In [12]:
list(raw_dir.glob("KAUST_Crossref_2024_2025_page_*.json"))

[WindowsPath('../data/raw/KAUST_Crossref_2024_2025_page_1.json'),
 WindowsPath('../data/raw/KAUST_Crossref_2024_2025_page_2.json')]